> ## SOLUTION / ANSWER KEY &mdash; Lab 9.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-assistive-not-autonomous.ipynb`](../lab-10-assistive-not-autonomous.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.10 &mdash; Assistive, Not Autonomous

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Return analysis flagged for review -- never a decision
- Require every claim to be cited so review is genuine
- Run a REAL model and gate its output as needs_review

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; Assistive on judgement, autonomous on legwork](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-10"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The golden rule of Day 5 (deck slides 11, 17): agents are **assistive, not autonomous**. The insight
agent does the grounded legwork and returns analysis flagged **`needs_review`** &mdash; it never decides.
And to defend against **automation bias** (humans rubber-stamping a confident machine), the review is only
real if the agent **shows its work**: every claim **cited**. The **human** owns any consequential decision.
Here you build the gate, then push a **real model's** output through it.

In [ ]:
# The agent's output is analysis + a needs_review flag -- never "executed" or a recommendation.
print("assistive output shape:", {"summary": "...", "status": "needs_review", "claims": ["..."]})

## Build it
Complete `make_insight` (flag for review), `reviewable` (require citations), and `owns_decision`, then
run the cell.

In [ ]:
def make_insight(summary, claims):
    # analysis + a needs_review flag -- the agent never decides
    return {"summary": summary, "status": "needs_review", "claims": claims}

def reviewable(insight):
    # a review is only genuine if EVERY claim is cited (defends against automation bias)
    return all(c.get("source") for c in insight["claims"])

def owns_decision(insight):
    # the agent only "owns" a decision it already executed; an assistive (needs_review) insight
    # is owned by a human -- so branch on the status field to return the owner
    return "agent" if insight["status"] == "executed" else "human"

try:
    claims = [{"metric": "revenue", "source": "p4"}, {"metric": "margin", "source": "p4"}]
    ins = make_insight("Revenue +12% YoY [p4]; margin down [p4].", claims)
    print("status    :", ins["status"])
    print("reviewable:", reviewable(ins))
    print("owns (assistive):", owns_decision(ins))
    print("owns (if executed):", owns_decision({"status": "executed"}))
    uncited = make_insight("...", [{"metric": "guess", "source": ""}])
    print("uncited reviewable?", reviewable(uncited))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Now ask the **real** Groq model for a cited, no-advice insight and push its output through your assistive gate &mdash; the model does the drafting, your code keeps it `needs_review` and advice-free.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. On the free tier, if you hit a rate limit wait a few seconds and re-run._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        # Ask the REAL model for a one-line, cited, NO-advice insight -- then push its output through YOUR gate.
        prompt = ("In ONE line, state revenue and net income with their page cites, and give NO advice: "
                  "revenue 120.0M (p4, income stmt), net income 9.0M (p4, income stmt).")
        text = llm.invoke(prompt).content
        print("REAL model insight:", text)
        ins = make_insight(text, [{"metric": "revenue", "source": "p4, income stmt"},
                                  {"metric": "net_income", "source": "p4, income stmt"}])
        print("status                  :", ins["status"], "(never a decision)")
        print("owner of the decision   :", owns_decision(ins))
        advice_terms = ("buy", "sell", "recommend", "you should", "invest in")
        print("no-advice guardrail holds:", not any(t in text.lower() for t in advice_terms))
    except Exception as e:
        print("(Rate-limited on the free tier? wait a few seconds. Or fill the ___ blanks, then re-run.)", type(e).__name__)

## What to notice
- The real model's text is wrapped **`needs_review`** &mdash; the agent never marks anything `executed`, so a **human** owns the decision.
- `reviewable` is only True when every claim is cited &mdash; an uncited insight makes review a rubber-stamp, and it's caught.
- The no-advice guardrail (Lab 5) runs on the **real** output here &mdash; a live model, gated by your deterministic rules.

## Your turn (open task &mdash; no grader)
Change the prompt to tempt the model into advice (e.g. add *"...and say whether to buy"*) and re-run.
**What good looks like:** even if the model slips, your `no-advice guardrail holds` check flips False so you
*catch* it &mdash; and the output is still `needs_review`, owned by a human. The safety is in your gate, not
the model's goodwill.

---
### Key takeaway
Assistive, not autonomous: the agent flags analysis for review and never decides, and citations make the review genuine rather than a rubber-stamp. You just gated a real model's output. The human owns every consequential decision.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>